In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"  # Ajustar se necessário

# ========== FILTRO DE RESTAURANTES E ESTABELECIMENTOS DE COMIDA ==========

# Tags de alimentação
FOOD_TAGS = {
    "Restaurants", "Food", "Fast Food", "Food Trucks", "Food Delivery Services",
    # Culinárias
    "Pizza", "Mexican", "Chinese", "Italian", "Japanese", "Thai", "Vietnamese",
    "Indian", "Greek", "Mediterranean", "French", "Korean", "Filipino", "African",
    "Cuban", "Caribbean", "Middle Eastern", "Latin American", "Asian Fusion",
    "American (Traditional)", "American (New)", "Canadian (New)", "Cajun/Creole",
    "Pakistani", "Southern", "Soul Food", "Tex-Mex",
    # Tipos de estabelecimento
    "Burgers", "Sandwiches", "Sushi Bars", "Seafood", "Steakhouses", "Barbeque",
    "Chicken Wings", "Chicken Shop", "Hot Dogs", "Cheesesteaks", "Tacos",
    "Delis", "Diners", "Buffets", "Breakfast & Brunch",
    "Cafes", "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
    "Juice Bars & Smoothies", "Desserts", "Bakeries", "Donuts", "Bagels",
    "Ice Cream & Frozen Yogurt", "Candy Stores",
    # Bares & bebidas
    "Bars", "Nightlife", "Breweries", "Wine & Spirits", "Beer", "Cocktail Bars",
    "Dive Bars", "Sports Bars", "Pubs", "Gastropubs", "Lounges",
    # Varejo alimentar
    "Grocery", "Specialty Food", "Seafood Markets", "Meat Shops", "Fruits & Veggies",
    "Farmers Market", "Convenience Stores", "Wholesale Stores",
    # Outros
    "Caterers",
}

def is_food_related(categories_str):
    """Verifica se o estabelecimento é relacionado a alimentação."""
    if not categories_str:
        return False
    tags = {t.strip() for t in str(categories_str).split(",")}
    return bool(tags & FOOD_TAGS)

def get_segmento_alimentacao(categories_str):
    """Retorna o segmento de alimentação baseado nas categorias."""
    if not categories_str:
        return "Outros alimentação"
    
    tags = {t.strip() for t in str(categories_str).split(",")}
    
    if "Restaurants" in tags:
        return "RESTAURANTE"
    if tags & {"Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
                "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer"}:
        return "BAR E BEBIDA"
    if tags & {"Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
                "Cafes", "Juice Bars & Smoothies"}:
        return "CAFE"
    if tags & {"Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
                "Candy Stores", "Desserts"}:
        return "PADARIA"
    # Se não se encaixa em nenhum segmento de alimentação, retorna "não são restaurantes"
    segmentos_alimentacao = {
        "Restaurants",
        "Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
        "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer",
        "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
        "Cafes", "Juice Bars & Smoothies",
        "Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
        "Candy Stores", "Desserts"
    }
    if not tags & segmentos_alimentacao:
        return "OUTROS"

# Registra UDFs para uso no Spark
is_food_udf = udf(is_food_related, BooleanType())
get_segmento_udf = udf(get_segmento_alimentacao, StringType())

# ========== FUNÇÕES DE QUALIDADE DE DADOS ==========

def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    """
    Calcula a taxa de completude (% de valores não nulos) para colunas obrigatórias.
    
    Dimensão de Qualidade: COMPLETUDE
    """
    total_registros = df.count()
    metricas = {}
    
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    """
    Valida se valores numéricos estão dentro de um range esperado.
    
    Dimensão de Qualidade: PRECISÃO
    """
    condicao = col(coluna).isNotNull()
    
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    """
    Remove duplicados mantendo o registro mais recente baseado no critério de desempate.
    """
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    """
    Adiciona metadados de processamento para camada silver.
    """
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))

print("✓ Funções de qualidade de dados carregadas com sucesso!")
print("✓ Filtros de restaurantes e alimentação configurados!")

In [0]:
# ========== TABELA 1: BUSINESS ==========

table_name = "bronze_yelp_academic_dataset_business"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_business = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_business.count()}")

# ========== FILTRO: ESTABELECIMENTOS DE ALIMENTAÇÃO ==========
print("\n--- Filtro de Estabelecimentos de Alimentação ---")
df_business = df_business.filter(is_food_udf(col('categories')))
print(f"Registros relacionados a alimentação: {df_business.count()}")

# Adiciona coluna de food_category
df_business = df_business.withColumn('food_category', get_segmento_udf(col('categories')))
print("✓ Coluna 'food_category' adicionada")

# Mostra distribuição por food_category
print("\nDistribuição por food_category:")
display(df_business.groupBy('food_category').count().orderBy(col('count').desc()))

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'name', 'city', 'state']
metricas_completude = calcular_completude(df_business, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_business_clean = df_business.filter(
    col('business_id').isNotNull() & 
    col('name').isNotNull() & 
    col('city').isNotNull() & 
    col('state').isNotNull()
)

print(f"\nApós filtro de completude: {df_business_clean.count()} registros")

# PRECISÃO: Validação de ranges numéricos
print("\n--- Validação de PRECISÃO ---")

# Stars: 0 a 5
df_business_clean = validar_precisao_numerica(df_business_clean, 'stars', min_val=0, max_val=5)
print(f"  Stars (0-5): {df_business_clean.count()} registros válidos")

# Latitude: -90 a 90
df_business_clean = validar_precisao_numerica(df_business_clean, 'latitude', min_val=-90, max_val=90)
print(f"  Latitude (-90/90): {df_business_clean.count()} registros válidos")

# Longitude: -180 a 180
df_business_clean = validar_precisao_numerica(df_business_clean, 'longitude', min_val=-180, max_val=180)
print(f"  Longitude (-180/180): {df_business_clean.count()} registros válidos")

# Review count: >= 0
df_business_clean = df_business_clean.filter(col('review_count') >= 0)
print(f"  Review count (>=0): {df_business_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_business_clean = remover_duplicados(df_business_clean, ['business_id'])
print(f"Após deduplication: {df_business_clean.count()} registros")

# ========== PADRONIZAÇÃO DO ENDEREÇO ==========
print("\n--- Padronização de Endereços ---")
from pyspark.sql.functions import upper, regexp_replace, trim

# Converte endereço para caixa alta
df_business_clean = df_business_clean.withColumn("address", upper(col("address")))
print("✓ Coluna 'address' convertida para caixa alta")

# ========== PADRONIZAÇÃO DO NOME ==========
print("\n--- Padronização de Nomes ---")

# Apply transformations step by step
name_col = upper(col("name"))
name_col = regexp_replace(name_col, r",.*", "")  # Remove text after comma
name_col = regexp_replace(name_col, r",", " ")  # Replace comma with space
name_col = regexp_replace(name_col, r"\.", " ")  # Replace dot with space
name_col = regexp_replace(name_col, r"ST\.", "SAINT")
name_col = regexp_replace(name_col, r"\bST\b", "SAINT")
name_col = regexp_replace(name_col, r"\bST\.\b", "SAINT")
name_col = regexp_replace(name_col, r"SAINTT", "SAINT")
name_col = regexp_replace(name_col, r"NW ", "NEW")
name_col = regexp_replace(name_col, r"'", " ")  # Replace apostrophes
name_col = regexp_replace(name_col, r"-", " ")  # Replace hyphens
name_col = regexp_replace(name_col, r"^ +| +$", "")  # Trim edges
name_col = regexp_replace(name_col, r" {2,}", " ")  # Collapse multiple spaces
name_col = trim(name_col)

df_business_clean = df_business_clean.withColumn("name_validated", name_col)
print("✓ Coluna 'name_validated' adicionada")

# ========== PADRONIZAÇÃO DA CIDADE ==========
print("\n--- Padronização de Cidades ---")

# Apply transformations step by step
city_col = upper(col("city"))
city_col = regexp_replace(city_col, r",.*", "")  # Remove content after comma
city_col = regexp_replace(city_col, r"/.*", "")  # Remove content after slash
city_col = regexp_replace(city_col, r"BCH", "BEACH")
city_col = regexp_replace(city_col, r",", " ")
city_col = regexp_replace(city_col, r"/", " ")
city_col = regexp_replace(city_col, r"%MTLAUREL%", "MT LAUREL")
city_col = regexp_replace(city_col, r"%TAMPA FLORIDA%", "TAMPA")
city_col = regexp_replace(city_col, r"TWP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"MT \.", "MT")
city_col = regexp_replace(city_col, r"MT\.", "MT")
city_col = regexp_replace(city_col, r"SAINTLOUIS", "SAINT LOUIS")
city_col = regexp_replace(city_col, r"SAINTT", "SAINT")
city_col = regexp_replace(city_col, r"^SAINT PETE$", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"SAINT PETERS", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"REDINGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"REDNGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"CHALEMETTE", "CHALMETTE")
city_col = regexp_replace(city_col, r"INPOLIS", "INDIANAPOLIS")
city_col = regexp_replace(city_col, r"CONSHOHOEKEN", "CONSHOHOCKEN")
city_col = regexp_replace(city_col, r"FESTERVILLE", "FEASTERVILLE")
city_col = regexp_replace(city_col, r"TOWSSHIP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"%TWN%", "TOWN")
city_col = regexp_replace(city_col, r"CNTRY", "COUNTRY")
city_col = regexp_replace(city_col, r"TIERRE VERDE", "TIERRA VERDE")
city_col = regexp_replace(city_col, r"NW", "NEW")
city_col = regexp_replace(city_col, r"TOWN & COUNTRY", "TOWN N COUNTRY")
city_col = regexp_replace(city_col, r"\.", " ")  # Replace dot with space
city_col = regexp_replace(city_col, r"ST\.", "SAINT")
city_col = regexp_replace(city_col, r"\bST\b", "SAINT")
city_col = regexp_replace(city_col, r"\bST\.\b", "SAINT")
city_col = regexp_replace(city_col, r"'", " ")  # Replace apostrophes
city_col = regexp_replace(city_col, r"-", " ")  # Replace hyphens
city_col = regexp_replace(city_col, r" {2,}", " ")  # Collapse multiple spaces
city_col = trim(city_col)  # Remove spaces at the beginning and end

df_business_clean = df_business_clean.withColumn("city_validated", city_col)
print("✓ Coluna 'city_validated' adicionada")

# ========== SUBSTITUIÇÃO DOS CAMPOS ORIGINAIS ==========
print("\n--- Substituição de Campos Originais ---")

# Remove campos originais name e city
df_business_clean = df_business_clean.drop('name', 'city')
print("✓ Campos originais 'name' e 'city' removidos")

# Renomeia os campos validados
df_business_clean = df_business_clean.withColumnRenamed('name_validated', 'name')
df_business_clean = df_business_clean.withColumnRenamed('city_validated', 'city')
print("✓ Campos renomeados: 'name_validated' -> 'name', 'city_validated' -> 'city'")

# Adiciona metadados silver
df_business_silver = adicionar_metadados_silver(df_business_clean)

# Remove coluna antiga 'segmento' se existir (para evitar duplicação com food_category)
if 'segmento' in df_business_silver.columns:
    df_business_silver = df_business_silver.drop('segmento')
    print("\n✓ Coluna antiga 'segmento' removida")

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_business"
df_business_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_business_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Business:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Métricas gerais
print(f"Total de registros: {df_sample.count()}")
print(f"Campos: {len(df_sample.columns)}")

# Amostra de dados
print("\nPrimeiros 10 registros:")
display(df_sample.select(
    'business_id', 
    'name',
    'city',
    'state', 
    'stars', 
    'review_count',
    'food_category',
    'data_processamento_silver'
).limit(10))

# Estatísticas de qualidade
print("\nEstatísticas de Stars:")
df_sample.select('stars').describe().show()

print("\nDistribuição por Estado (Top 10):")
display(df_sample.groupBy('state').count().orderBy(col('count').desc()).limit(10))

In [0]:
%sql
SELECT 
  name,
  address,
  city,
  state,
  food_category
FROM workspace.yelp_ing.silver_business
WHERE address IS NOT NULL
LIMIT 10

In [0]:
# ========== AMOSTRAS DE PADRONIZAÇÃO DE NOMES ==========

print("ANÁLISE DE PADRONIZAÇÃO DE NOMES")
print("="*80)

df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Amostra geral
print("\n1. Amostra Geral (20 registros):")
print("-"*80)
display(df_business.select('name', 'city', 'state').limit(20))

# Estatísticas
print("\n2. Estatísticas de Transformação:")
print("-"*80)
total = df_business.count()

print(f"Total de estabelecimentos: {total:,}")
print(f"Campos 'name' e 'city' padronizados com sucesso!")

print("\n✓ Análise de padronização concluída!")

In [0]:
# ========== TABELA 2: REVIEW ==========

table_name = "bronze_yelp_academic_dataset_review"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_review = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_review.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['review_id', 'user_id', 'business_id', 'stars', 'text', 'date']
metricas_completude = calcular_completude(df_review, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_review_clean = df_review.filter(
    col('review_id').isNotNull() & 
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() &
    col('stars').isNotNull() &
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_review_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Stars: 1 a 5 (reviews não permitem 0 stars)
df_review_clean = validar_precisao_numerica(df_review_clean, 'stars', min_val=1, max_val=5)
print(f"  Stars (1-5): {df_review_clean.count()} registros válidos")

# Texto não vazio (após trim)
df_review_clean = df_review_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_review_clean.count()} registros válidos")

# Useful, funny, cool >= 0
for coluna in ['useful', 'funny', 'cool']:
    df_review_clean = df_review_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (useful/funny/cool >=0): {df_review_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_review_clean = remover_duplicados(df_review_clean, ['review_id'])
print(f"Após deduplication: {df_review_clean.count()} registros")

# Adiciona metadados silver
df_review_silver = adicionar_metadados_silver(df_review_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_review"
df_review_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_review_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Review:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")

print(f"Total de reviews: {df_sample.count()}")

# Amostra de dados
print("\nPrimeiros 5 registros:")
display(df_sample.select(
    'review_id', 
    'user_id', 
    'business_id', 
    'stars',
    'date',
    'data_processamento_silver'
).limit(5))

# Distribuição de stars
print("\nDistribuição de Stars:")
display(df_sample.groupBy('stars').count().orderBy('stars'))

In [0]:
# ========== TABELA 3: USER ==========

table_name = "bronze_yelp_academic_dataset_user"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_user = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_user.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'name', 'yelping_since']
metricas_completude = calcular_completude(df_user, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_user_clean = df_user.filter(
    col('user_id').isNotNull() & 
    col('name').isNotNull() & 
    col('yelping_since').isNotNull()
)

print(f"\nApós filtro de completude: {df_user_clean.count()} registros")

# PRECISÃO: Validações numéricas
print("\n--- Validação de PRECISÃO ---")

# Average stars: 0 a 5
df_user_clean = validar_precisao_numerica(df_user_clean, 'average_stars', min_val=0, max_val=5)
print(f"  Average stars (0-5): {df_user_clean.count()} registros válidos")

# Review count, fans, useful, funny, cool >= 0
for coluna in ['review_count', 'fans', 'useful', 'funny', 'cool']:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (>=0): {df_user_clean.count()} registros válidos")

# Compliments >= 0
compliment_cols = [c for c in df_user_clean.columns if c.startswith('compliment_')]
for coluna in compliment_cols:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Compliments (>=0): {df_user_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_user_clean = remover_duplicados(df_user_clean, ['user_id'])
print(f"Após deduplication: {df_user_clean.count()} registros")

# Adiciona metadados silver
df_user_silver = adicionar_metadados_silver(df_user_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_user"
df_user_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_user_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 4: TIP ==========

table_name = "bronze_yelp_academic_dataset_tip"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_tip = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_tip.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'business_id', 'text', 'date']
metricas_completude = calcular_completude(df_tip, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_tip_clean = df_tip.filter(
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() & 
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_tip_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Texto não vazio
df_tip_clean = df_tip_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_tip_clean.count()} registros válidos")

# Compliment count >= 0
df_tip_clean = df_tip_clean.filter(
    col('compliment_count').isNull() | (col('compliment_count') >= 0)
)
print(f"  Compliment count (>=0): {df_tip_clean.count()} registros válidos")

# Remoção de duplicados (chave composta: user_id + business_id + date + text)
print("\n--- Remoção de Duplicados ---")
df_tip_clean = remover_duplicados(df_tip_clean, ['user_id', 'business_id', 'date', 'text'])
print(f"Após deduplication: {df_tip_clean.count()} registros")

# Adiciona metadados silver
df_tip_silver = adicionar_metadados_silver(df_tip_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_tip"
df_tip_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_tip_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 5: CHECKIN ==========

table_name = "bronze_yelp_academic_dataset_checkin"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_checkin = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_checkin.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'date']
metricas_completude = calcular_completude(df_checkin, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_checkin_clean = df_checkin.filter(
    col('business_id').isNotNull() & 
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_checkin_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Date não vazio (campo contém timestamps separados por vírgula)
df_checkin_clean = df_checkin_clean.filter(length(trim(col('date'))) > 0)
print(f"  Date não vazio: {df_checkin_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_checkin_clean = remover_duplicados(df_checkin_clean, ['business_id'])
print(f"Após deduplication: {df_checkin_clean.count()} registros")

# Adiciona metadados silver
df_checkin_silver = adicionar_metadados_silver(df_checkin_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_checkin"
df_checkin_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_checkin_silver.count()}")
print("="*60)

In [0]:
# ========== RESUMO FINAL: QUALIDADE DE DADOS SILVER ==========

print("RELATÓRIO DE QUALIDADE - CAMADA SILVER")
print("="*80)
print("\nDimensões de Qualidade Aplicadas:")
print("  1. COMPLETUDE: Campos obrigatórios preenchidos")
print("  2. PRECISÃO: Valores dentro dos ranges esperados\n")
print("="*80)

# Lista de tabelas silver
tabelas_silver = [
    'silver_business',
    'silver_review',
    'silver_user',
    'silver_tip',
    'silver_checkin'
]

resumo = []

for tabela in tabelas_silver:
    try:
        df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{tabela}")
        count = df.count()
        colunas = len(df.columns)
        
        # Verifica se tem o campo de processamento
        tem_metadata = 'data_processamento_silver' in df.columns
        
        resumo.append({
            'Tabela': tabela,
            'Registros': count,
            'Colunas': colunas,
            'Metadata Silver': '✓' if tem_metadata else '✗'
        })
        
    except Exception as e:
        print(f"Erro ao processar {tabela}: {str(e)}")

# Cria DataFrame de resumo
import pandas as pd
df_resumo = pd.DataFrame(resumo)

print("\nTABELAS SILVER CRIADAS:")
print(df_resumo.to_string(index=False))

print("\n" + "="*80)
print("\nPRÓXIMOS PASSOS RECOMENDADOS:")
print("  1. Validar relacionamentos entre tabelas (FKs)")
print("  2. Criar testes de qualidade automatizados")
print("  3. Implementar monitoramento contínuo")
print("  4. Documentar regras de negócio aplicadas")
print("  5. Criar camada Gold com agregações analíticas")
print("\n" + "="*80)
print("\n✓ Processo de limpeza concluído com sucesso!")

In [0]:
from pyspark.sql.functions import date_format, dayofweek

def gerar_tabela_business_reviews():
    """
    Gera uma tabela integrada com dados de estabelecimentos e reviews.
    
    Retorna:
        DataFrame com colunas:
        - estabelecimento (nome)
        - food_category
        - review_id
        - stars (avaliação)
        - text (texto do review)
        - date (data do review)
        - dia_semana (nome do dia da semana)
        - dia_semana_numero (1=Domingo, 2=Segunda, ..., 7=Sábado)
        - estado (localização do estabelecimento)
    """
    
    print("Gerando tabela integrada Business + Reviews...")
    print("="*60)
    
    # Carrega tabelas silver
    df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")
    df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")
    
    print(f"Estabelecimentos: {df_business.count():,} registros")
    print(f"Reviews: {df_review.count():,} registros")
    
    # JOIN entre business e review
    df_integrado = df_review.join(
        df_business,
        df_review.business_id == df_business.business_id,
        'inner'
    )
    
    print(f"\nRegistros após JOIN: {df_integrado.count():,}")
    
    # Extrai dia da semana do review
    df_integrado = df_integrado.withColumn(
        'dia_semana_numero',
        dayofweek(col('date'))  # 1=Domingo, 2=Segunda, ..., 7=Sábado
    )
    
    # Converte número para nome do dia
    df_integrado = df_integrado.withColumn(
        'dia_semana',
        date_format(col('date'), 'EEEE')  # Nome completo do dia em inglês
    )
    
    # Seleciona e renomeia colunas
    df_resultado = df_integrado.select(
        df_business.name.alias('estabelecimento'),
        df_business.food_category,
        df_review.review_id,
        df_review.stars,
        df_review.text,
        df_review.date,
        col('dia_semana'),
        col('dia_semana_numero'),
        df_business.state.alias('estado')
    )
    
    print("\n✓ Tabela integrada gerada com sucesso!")
    print("="*60)
    
    return df_resultado

# Testa a função
df_tabela_integrada = gerar_tabela_business_reviews()

# Mostra amostra
print("\nAMOSTRA DA TABELA INTEGRADA (10 primeiros registros):")
print("-"*60)
display(df_tabela_integrada.limit(10))

# Estatísticas
print("\nDISTRIBUIÇÃO POR DIA DA SEMANA:")
display(
    df_tabela_integrada
    .groupBy('dia_semana', 'dia_semana_numero')
    .count()
    .orderBy('dia_semana_numero')
)

print("\nDISTRIBUIÇÃO POR ESTADO (Top 10):")
display(
    df_tabela_integrada
    .groupBy('estado')
    .count()
    .orderBy(col('count').desc())
    .limit(10)
)

print("\nDISTRIBUIÇÃO POR CATEGORIA DE COMIDA:")
display(
    df_tabela_integrada
    .groupBy('food_category')
    .count()
    .orderBy(col('count').desc())
)

In [0]:
# ========== SALVAR TABELA INTEGRADA COMO SILVER ==========

print("Salvando tabela integrada na camada Silver...")
print("="*60)

# Nome da tabela silver
table_name = f"{CATALOG}.{SCHEMA_SILVER}.silver_business_reviews_integrado"

# Adiciona metadados de processamento
df_tabela_final = df_tabela_integrada.withColumn(
    'data_processamento_silver',
    current_timestamp()
)

print(f"\nTotal de registros a salvar: {df_tabela_final.count():,}")
print(f"Campos: {len(df_tabela_final.columns)}")

# Salva a tabela
df_tabela_final.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela silver salva com sucesso: {table_name}")
print("="*60)

# Verifica a tabela criada
print("\nVerificando tabela criada:")
df_verificacao = spark.table(table_name)

print(f"Total de registros na tabela: {df_verificacao.count():,}")
print(f"\nSchema da tabela:")
df_verificacao.printSchema()

# Mostra amostra
print("\nAmostra de 5 registros:")
display(df_verificacao.limit(5))

In [0]:
# ========== GRÁFICO: REVIEWS POR DIA DA SEMANA E CATEGORIA ==========

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("Gerando gráfico de reviews por dia da semana e categoria...")
print("="*60)

# Carrega a tabela integrada
df_integrado = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business_reviews_integrado")

# Agrupa por food_category e dia da semana
df_agrupado = df_integrado.groupBy('food_category', 'dia_semana', 'dia_semana_numero') \
    .count() \
    .orderBy('dia_semana_numero', 'food_category')

# Converte para pandas para facilitar a visualização
df_pandas = df_agrupado.toPandas()

print(f"Total de registros agrupados: {len(df_pandas)}")
print("\nAmostra dos dados:")
print(df_pandas.head(10))

# Ordena os dias da semana corretamente
dias_ordem = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
dias_pt = ['Domingo', 'Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado']

# Pivot para facilitar o gráfico
df_pivot = df_pandas.pivot(index='dia_semana', columns='food_category', values='count')
df_pivot = df_pivot.reindex(dias_ordem)
df_pivot = df_pivot.fillna(0)

print("\nDados pivotados:")
print(df_pivot)

# Configuração do gráfico
fig, ax = plt.subplots(figsize=(14, 8))

# Cores para cada categoria
cores = {
    'RESTAURANTE': '#FF6B6B',
    'BAR E BEBIDA': '#4ECDC4',
    'CAFE': '#FFE66D',
    'PADARIA': '#95E1D3',
    'OUTROS': '#A8E6CF'
}

# Posição das barras
x = np.arange(len(dias_pt))
width = 0.15  # Largura das barras

# Plota cada categoria
for i, categoria in enumerate(df_pivot.columns):
    offset = width * (i - len(df_pivot.columns)/2 + 0.5)
    cor = cores.get(categoria, '#CCCCCC')
    ax.bar(x + offset, df_pivot[categoria], width, label=categoria, color=cor, alpha=0.8)

# Configurações do gráfico
ax.set_xlabel('Dia da Semana', fontsize=12, fontweight='bold')
ax.set_ylabel('Quantidade de Reviews', fontsize=12, fontweight='bold')
ax.set_title('Distribuição de Reviews por Dia da Semana e Categoria de Comida', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(dias_pt, fontsize=10)
ax.legend(title='Categoria', fontsize=10, title_fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Adiciona valores nas barras (opcional - pode deixar o gráfico poluído)
# for container in ax.containers:
#     ax.bar_label(container, fmt='%.0f', fontsize=7, padding=2)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("✓ Gráfico gerado com sucesso!")
print("\nINSIGHTS:")
print("  • Domingo e Sábado tendem a ter mais reviews")
print("  • Restaurantes dominam em todos os dias da semana")
print("  • Quinta-feira tem o menor volume de reviews")
print("="*60)

In [0]:
# ========== ADICIONAR MACRORREGIÃO BASEADA NO CENSO DOS EUA ==========

from pyspark.sql.functions import when, col

print("Adicionando coluna de macrorregião às tabelas...")
print("="*80)

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"

# Mapeamento de estados para macrorregiões (Censo dos EUA)
MAPA_MACROREGIAO = {
    # NORDESTE
    'CT': 'Nordeste', 'ME': 'Nordeste', 'MA': 'Nordeste', 'NH': 'Nordeste', 
    'RI': 'Nordeste', 'VT': 'Nordeste',  # Nova Inglaterra
    'NJ': 'Nordeste', 'NY': 'Nordeste', 'PA': 'Nordeste',  # Médio Atlântico
    
    # CENTRO-OESTE
    'IL': 'Centro-Oeste', 'IN': 'Centro-Oeste', 'MI': 'Centro-Oeste', 
    'OH': 'Centro-Oeste', 'WI': 'Centro-Oeste',  # Leste Norte-Central
    'IA': 'Centro-Oeste', 'KS': 'Centro-Oeste', 'MN': 'Centro-Oeste', 
    'MO': 'Centro-Oeste', 'NE': 'Centro-Oeste', 'ND': 'Centro-Oeste', 
    'SD': 'Centro-Oeste',  # Oeste Norte-Central
    
    # SUL
    'DE': 'Sul', 'FL': 'Sul', 'GA': 'Sul', 'MD': 'Sul', 'NC': 'Sul', 
    'SC': 'Sul', 'VA': 'Sul', 'WV': 'Sul', 'DC': 'Sul',  # Atlântico Sul
    'AL': 'Sul', 'KY': 'Sul', 'MS': 'Sul', 'TN': 'Sul',  # Leste Sul-Central
    'AR': 'Sul', 'LA': 'Sul', 'OK': 'Sul', 'TX': 'Sul',  # Oeste Sul-Central
    
    # OESTE
    'AZ': 'Oeste', 'CO': 'Oeste', 'ID': 'Oeste', 'MT': 'Oeste', 
    'NV': 'Oeste', 'NM': 'Oeste', 'UT': 'Oeste', 'WY': 'Oeste',  # Montanha
    'AK': 'Oeste', 'CA': 'Oeste', 'HI': 'Oeste', 'OR': 'Oeste', 
    'WA': 'Oeste',  # Pacífico
    
    # CANADÁ (estados fora dos EUA)
    'AB': 'Canadá', 'BC': 'Canadá', 'ON': 'Canadá', 'QC': 'Canadá'
}

# Função para mapear estado -> macrorregião
def adicionar_coluna_macroregiao(df, coluna_estado='estado'):
    """
    Adiciona coluna 'macrorregiao' baseada no estado.
    """
    # Cria expressão CASE WHEN para todos os estados
    expr_macroregiao = None
    for estado, macroregiao in MAPA_MACROREGIAO.items():
        if expr_macroregiao is None:
            expr_macroregiao = when(col(coluna_estado) == estado, macroregiao)
        else:
            expr_macroregiao = expr_macroregiao.when(col(coluna_estado) == estado, macroregiao)
    
    # Default para estados não mapeados
    expr_macroregiao = expr_macroregiao.otherwise('Não classificado')
    
    return df.withColumn('macrorregiao', expr_macroregiao)

# ========== ATUALIZAR TABELA: silver_business_reviews_integrado ==========

print("\n1. Atualizando silver_business_reviews_integrado...")
table1 = f"{CATALOG}.{SCHEMA_SILVER}.silver_business_reviews_integrado"
df1 = spark.table(table1)

print(f"   Registros antes: {df1.count():,}")
print(f"   Colunas antes: {len(df1.columns)}")

# Adiciona macrorregião
df1_atualizado = adicionar_coluna_macroregiao(df1)

print(f"   Colunas depois: {len(df1_atualizado.columns)}")

# Salva tabela atualizada (com overwriteSchema para permitir adicionar coluna)
df1_atualizado.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table1)

print(f"   ✓ Tabela atualizada: {table1}")

# Mostra distribuição por macrorregião
print("\n   Distribuição por Macrorregião:")
df1_verificacao = spark.table(table1)
display(
    df1_verificacao
    .groupBy('macrorregiao')
    .count()
    .orderBy(col('count').desc())
)

# ========== ATUALIZAR TABELA: silver_top10_estabelecimentos ==========

print("\n2. Atualizando silver_top10_estabelecimentos...")
table2 = f"{CATALOG}.{SCHEMA_SILVER}.silver_top10_estabelecimentos"
df2 = spark.table(table2)

print(f"   Registros antes: {df2.count():,}")
print(f"   Colunas antes: {len(df2.columns)}")

# Adiciona macrorregião
df2_atualizado = adicionar_coluna_macroregiao(df2)

print(f"   Colunas depois: {len(df2_atualizado.columns)}")

# Salva tabela atualizada (com overwriteSchema para permitir adicionar coluna)
df2_atualizado.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table2)

print(f"   ✓ Tabela atualizada: {table2}")

# Mostra distribuição por macrorregião
print("\n   Distribuição por Macrorregião:")
df2_verificacao = spark.table(table2)
display(
    df2_verificacao
    .groupBy('macrorregiao')
    .count()
    .orderBy(col('count').desc())
)

print("\n" + "="*80)
print("✓ Coluna 'macrorregiao' adicionada com sucesso às tabelas!")
print("="*80)

# Mostra amostra com a nova coluna
print("\nAMOSTRA com macrorregião (silver_business_reviews_integrado):")
display(
    df1_verificacao.select(
        'estabelecimento', 'food_category', 'estado', 'macrorregiao', 'stars'
    ).limit(10)
)

In [0]:
# ========== TOP 10 RESTAURANTES POR SEGMENTO, ESTADO E ANO ==========

from pyspark.sql.functions import avg, count, desc, row_number, col, year
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"

print("Analisando os melhores restaurantes por segmento, localidade e ano...")
print("="*80)

# Carrega tabela integrada
df_integrado = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business_reviews_integrado")

print(f"Total de reviews na base: {df_integrado.count():,}")

# Extrai ano da data do review
df_integrado = df_integrado.withColumn('ano', year(col('date')))

# Agrupa por estabelecimento, categoria, estado, macrorregiao e ano
df_ranking = df_integrado.groupBy('estabelecimento', 'food_category', 'estado', 'macrorregiao', 'ano').agg(
    avg('stars').alias('media_stars'),
    count('review_id').alias('total_reviews')
)

# Filtra apenas estabelecimentos com pelo menos 50 reviews (para ter relevância estatística)
df_ranking = df_ranking.filter(col('total_reviews') >= 50)

print(f"\nEstabelecimentos com 50+ reviews: {df_ranking.count():,}")

# Cria window para rankear dentro de cada categoria + estado + ano
window_spec = Window.partitionBy('food_category', 'estado', 'ano').orderBy(desc('media_stars'), desc('total_reviews'))

# Adiciona ranking
df_ranking = df_ranking.withColumn('ranking', row_number().over(window_spec))

# Filtra apenas os top 10 de cada grupo
df_top10 = df_ranking.filter(col('ranking') <= 10)

print(f"\nTotal de registros no top 10: {df_top10.count():,}")

# Ordena o resultado final
df_top10_final = df_top10.orderBy('ano', 'food_category', 'estado', 'ranking')

print("\n✓ Análise concluída!")
print("="*80)

# Mostra resumo das combinações disponíveis
print("\nCOMBINAÇÕES DISPONÍVEIS (Categoria + Estado + Ano):")
df_combinacoes = df_top10_final.select('food_category', 'estado', 'ano').distinct().orderBy('ano', 'food_category', 'estado')
print(f"Total de combinações: {df_combinacoes.count()}")
print(f"\nAnos disponíveis: {df_top10_final.select('ano').distinct().count()}")
display(df_top10_final.select('ano').distinct().orderBy('ano'))

# Armazena o resultado
df_top10_cached = df_top10_final
print(f"\n✓ DataFrame com top 10 restaurantes criado: df_top10_cached")

In [0]:
# ========== EXEMPLOS DE TOP 10 POR CATEGORIA E ESTADO ==========

from pyspark.sql.functions import col

print("EXEMPLOS DE RANKINGS")
print("="*80)

# Exemplo 1: Top 10 RESTAURANTES na Pensilvânia (PA)
print("\n1. TOP 10 RESTAURANTES EM PENNSYLVANIA (PA):")
print("-"*80)
df_exemplo1 = df_top10_cached.filter(
    (col('food_category') == 'RESTAURANTE') & (col('estado') == 'PA')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Restaurante'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo1)

# Exemplo 2: Top 10 BARES na Flórida (FL)
print("\n2. TOP 10 BARES E BEBIDAS EM FLORIDA (FL):")
print("-"*80)
df_exemplo2 = df_top10_cached.filter(
    (col('food_category') == 'BAR E BEBIDA') & (col('estado') == 'FL')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Bar'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo2)

# Exemplo 3: Top 10 CAFES no Arizona (AZ)
print("\n3. TOP 10 CAFES EM ARIZONA (AZ):")
print("-"*80)
df_exemplo3 = df_top10_cached.filter(
    (col('food_category') == 'CAFE') & (col('estado') == 'AZ')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Café'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo3)

print("\n" + "="*80)
print("✓ Use df_top10_cached.filter() para consultar outras combinações!")
print("="*80)

In [0]:
# ========== SALVAR TABELA TOP 10 COMO SILVER ==========

from pyspark.sql.functions import current_timestamp

# Configuração
CATALOG = "workspace"
SCHEMA_SILVER = "yelp_ing"

print("Salvando tabela de top 10 restaurantes na camada Silver...")
print("="*80)

# Nome da tabela
table_name = f"{CATALOG}.{SCHEMA_SILVER}.silver_top10_estabelecimentos"

# Adiciona metadados
df_top10_final_com_metadata = df_top10_cached.withColumn(
    'data_processamento',
    current_timestamp()
)

print(f"\nTotal de registros: {df_top10_final_com_metadata.count():,}")
print(f"Campos: {len(df_top10_final_com_metadata.columns)}")

# Salva a tabela (com overwriteSchema para permitir adicionar coluna ano)
df_top10_final_com_metadata.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela salva com sucesso: {table_name}")
print("="*80)

# Verificação
print("\nVerificando tabela criada:")
df_verificacao = spark.table(table_name)
print(f"Total de registros: {df_verificacao.count():,}")

print("\nSchema:")
df_verificacao.printSchema()

# Mostra distribuição por ano
print("\nDistribuição por Ano:")
display(
    df_verificacao
    .groupBy('ano')
    .count()
    .orderBy('ano')
)

# Mostra distribuição por macrorregião e ano
print("\nDistribuição por Macrorregião e Ano (Top 5 anos):")
display(
    df_verificacao
    .groupBy('macrorregiao', 'ano')
    .count()
    .orderBy(col('ano').desc(), col('count').desc())
    .limit(20)
)

print("\nAmostra (5 registros):")
display(df_verificacao.limit(5))

print("\n" + "="*80)
print("PRÓXIMOS PASSOS:")
print("  1. Criar dashboard com os top 10 por categoria, macrorregião e ano")
print("  2. Adicionar filtros interativos de categoria e ano")
print("  3. Criar 5 tabelas (uma para cada macrorregião)")
print("  4. Criar gráfico de rosca com distribuição por macrorregião")
print("="*80)